In [1]:
# !pip install datasets
!pip install transformers
!pip install torch

In [6]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import f1_score, accuracy_score

In [3]:
train_path = "training_data/train.csv"
dev_path = "training_data//dev.csv"

In [7]:
train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)


In [8]:
deberta_name = "microsoft/deberta-v3-large"
modernbert_name = "answerdotai/ModernBERT-base"

In [9]:
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
modern_tokenizer = AutoTokenizer.from_pretrained(modernbert_name)

MAX_LEN = 256

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [10]:
def tokenize_deberta(example):
    return deberta_tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )


def tokenize_modern(example):
    return modern_tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

In [11]:

train_deberta = train_dataset.map(tokenize_deberta, batched=True)
dev_deberta = dev_dataset.map(tokenize_deberta, batched=True)

train_modern = train_dataset.map(tokenize_modern, batched=True)
dev_modern = dev_dataset.map(tokenize_modern, batched=True)

train_deberta.set_format(type="torch",
                         columns=["input_ids", "attention_mask", "label"])

dev_deberta.set_format(type="torch",
                       columns=["input_ids", "attention_mask", "label"])

train_modern.set_format(type="torch",
                        columns=["input_ids", "attention_mask", "label"])

dev_modern.set_format(type="torch",
                      columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

In [14]:
training_args_deberta = TrainingArguments(

    output_dir="deberta_results",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_dir="logs_deberta",

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
training_args_modern = TrainingArguments(

    output_dir="modernbert_results",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_dir="logs_modern",

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name,
    num_labels=2
)

modern_model = AutoModelForSequenceClassification.from_pretrained(
    modernbert_name,
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
trainer_deberta = Trainer(

    model=deberta_model,
    args=training_args_deberta,

    train_dataset=train_deberta,
    eval_dataset=dev_deberta,

    compute_metrics=compute_metrics
)

trainer_modern = Trainer(

    model=modern_model,
    args=training_args_modern,

    train_dataset=train_modern,
    eval_dataset=dev_modern,

    compute_metrics=compute_metrics
)

In [20]:
print("\nTraining DeBERTa...")
trainer_deberta.train()

print("\nTraining ModernBERT...")
trainer_modern.train()



Training DeBERTa...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print("\nEvaluating DeBERTa")
deberta_results = trainer_deberta.evaluate()

print("\nEvaluating ModernBERT")
modern_results = trainer_modern.evaluate()

In [ ]:
print("\nRunning ensemble evaluation...")

pred_deberta = trainer_deberta.predict(dev_deberta)
pred_modern = trainer_modern.predict(dev_modern)

logits_deberta = pred_deberta.predictions
logits_modern = pred_modern.predictions

# Average logits
final_logits = (logits_deberta + logits_modern) / 2

final_preds = np.argmax(final_logits, axis=1)

macro_f1 = f1_score(dev_df["label"], final_preds, average="macro")
accuracy = accuracy_score(dev_df["label"], final_preds)

print("\n=============================")
print("ENSEMBLE RESULTS")
print("=============================")
print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)